# Attack Evaluation: Privacy Analysis

This notebook provides comprehensive privacy evaluation for the obfuscation framework.
It runs all implemented attacks (MI-FGSM, L-BFGS, VAE, CycleGAN) against obfuscated images
and includes a tanh ablation study to assess how the activation function affects privacy guarantees.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "vit_obfuscation").exists():
            return path
    raise RuntimeError("Could not find repository root. Start Jupyter inside the repository.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_DIR = ROOT / "configs" / "experiments"
CANONICAL_OUTPUT_DIR = ROOT / "outputs" / "revision_v3"
NOTEBOOK_OUTPUT_DIR = ROOT / "outputs" / "notebook_runs"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Canonical artifacts: {CANONICAL_OUTPUT_DIR}")
print(f"Notebook outputs: {NOTEBOOK_OUTPUT_DIR}")

In [ ]:
import torch
import logging
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Load Model and Prepare Data

In [ ]:
from vit_obfuscation.config.experiment import ExperimentConfig
from transformers import AutoModel, AutoProcessor
from vit_obfuscation.adapter.model_adapter import ModelAdapter
from vit_obfuscation.obfuscation.obfuscator import ChannelWisePatchLevelObfuscator
from vit_obfuscation.embedding.embedding import ObfuscationEmbedding
from vit_obfuscation.runner.runner import load_checkpoint, _checkpoint_key
from datasets import load_dataset

config = ExperimentConfig.from_yaml(str(CONFIG_DIR / "vit_cifar10.yaml"))
config.output_dir = str(NOTEBOOK_OUTPUT_DIR)
model = AutoModel.from_pretrained(config.model.hf_model_name_or_path)
processor = AutoProcessor.from_pretrained(config.model.hf_model_name_or_path)
adapter = ModelAdapter(model)
vision_config = adapter.get_vision_config()

obfuscator = ChannelWisePatchLevelObfuscator(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=config.obfuscation.patch_size,
    group_size=config.obfuscation.group_size,
    apply_patch_permutation=config.obfuscation.apply_patch_permutation,
)
obf_embedding = ObfuscationEmbedding(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=vision_config.patch_size,
    embed_dim=vision_config.hidden_size,
    num_extra_tokens=adapter.get_num_extra_tokens(),
)
adapter.copy_frozen_params(obf_embedding)
ckpt_key = _checkpoint_key(config)
if load_checkpoint(obfuscator, obf_embedding, config.output_dir, ckpt_key):
    print("Loaded notebook checkpoint")
else:
    print("No notebook checkpoint found; using the seeded obfuscator state for attack visualization")
print("Model and obfuscation modules loaded")

## Step 2: Prepare Attack Data

In [ ]:
dataset = load_dataset(config.dataset.hf_dataset_name_or_path, split=config.dataset.eval_split)
N = 32
images = [dataset[i][config.dataset.input_column].convert("RGB") for i in range(N)]
inputs = processor(images=images, return_tensors="pt")
pixel_values = inputs["pixel_values"]

with torch.no_grad():
    obfuscated = obfuscator(pixel_values)
print(f"Prepared {N} images: original {pixel_values.shape}, obfuscated {obfuscated.shape}")

## Step 3: Run All Attacks

In [ ]:
import pandas as pd
from vit_obfuscation.attacks.evaluate_attacks import evaluate_all_attacks

results = evaluate_all_attacks(
    obfuscator, pixel_values, obfuscated,
    unet_epochs=50, vae_epochs=50, cyclegan_epochs=100,
)

df = pd.DataFrame([{"Attack": r.attack_name, "SSIM": f"{r.ssim:.4f}", "PSNR (dB)": f"{r.psnr:.2f}"} for r in results])
print(df.to_string(index=False))

## Step 4: Visual Comparison

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
col_titles = ["Original", "Obfuscated", "MI-FGSM", "L-BFGS", "VAE", "CycleGAN"]
attack_key = {
    "MI-FGSM": "MI-FGSM",
    "L-BFGS": "L-BFGS",
    "VAE": "Adversarial VAE",
    "CycleGAN": "CycleGAN",
}

# Collect reconstructed images from each attack.
attack_images = {r.attack_name: r.reconstructed for r in results if r.reconstructed is not None}

for row in range(4):
    # Original
    img = pixel_values[row].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    axes[row, 0].imshow(img)
    axes[row, 0].axis("off")

    # Obfuscated
    img = obfuscated[row].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    axes[row, 1].imshow(img)
    axes[row, 1].axis("off")

    # Attack reconstructions
    for col, title in enumerate(["MI-FGSM", "L-BFGS", "VAE", "CycleGAN"], start=2):
        attack_name = attack_key[title]
        if attack_name in attack_images:
            img = attack_images[attack_name][row].permute(1, 2, 0).detach().numpy()
            img = (img - img.min()) / (img.max() - img.min())
            axes[row, col].imshow(img)
        axes[row, col].axis("off")

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title)

plt.tight_layout()
plt.show()

## Step 5: Quantitative Summary

In [ ]:
# Filter out baseline entry
attack_results = [r for r in results if "no attack" not in r.attack_name.lower()]
attack_names = [r.attack_name for r in attack_results]
ssim_values = [r.ssim for r in attack_results]
psnr_values = [r.psnr for r in attack_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(attack_names, ssim_values, color="steelblue")
ax1.set_ylabel("SSIM")
ax1.set_title("SSIM per Attack")
ax1.set_ylim(0, 1)
ax1.tick_params(axis="x", rotation=15)

ax2.bar(attack_names, psnr_values, color="coral")
ax2.set_ylabel("PSNR (dB)")
ax2.set_title("PSNR per Attack")
ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## Step 6: Tanh Ablation Study

Compare attack effectiveness with and without the tanh activation in the obfuscator.

In [ ]:
obfuscator_no_tanh = ChannelWisePatchLevelObfuscator(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=config.obfuscation.patch_size,
    group_size=config.obfuscation.group_size,
    use_tanh=False,
    apply_patch_permutation=config.obfuscation.apply_patch_permutation,
)

with torch.no_grad():
    obfuscated_no_tanh = obfuscator_no_tanh(pixel_values)

results_no_tanh = evaluate_all_attacks(
    obfuscator_no_tanh, pixel_values, obfuscated_no_tanh,
    unet_epochs=50, vae_epochs=50, cyclegan_epochs=100,
)

df_comparison = pd.DataFrame([
    {"Variant": "With tanh", "Attack": r.attack_name, "SSIM": r.ssim, "PSNR": r.psnr}
    for r in results
] + [
    {"Variant": "Without tanh", "Attack": r.attack_name, "SSIM": r.ssim, "PSNR": r.psnr}
    for r in results_no_tanh
])
print(df_comparison.pivot_table(index="Attack", columns="Variant", values=["SSIM", "PSNR"]).to_string())

## Step 7: Ablation Visualization

In [ ]:
import numpy as np

# Filter out baseline entries
ablation_with = [r for r in results if "no attack" not in r.attack_name.lower()]
ablation_without = [r for r in results_no_tanh if "no attack" not in r.attack_name.lower()]
attack_names_abl = [r.attack_name for r in ablation_with]

x = np.arange(len(attack_names_abl))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width / 2, [r.ssim for r in ablation_with], width, label="With tanh", color="steelblue")
bars2 = ax.bar(x + width / 2, [r.ssim for r in ablation_without], width, label="Without tanh", color="coral")

ax.set_ylabel("SSIM")
ax.set_title("Tanh Ablation: SSIM per Attack")
ax.set_xticks(x)
ax.set_xticklabels(attack_names_abl)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()